In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l6.3 SwiGLU

> 🎯 **Goal:** Upgrade the feed-forward block from a plain MLP to SwiGLU, which adds a
learned multiplicative gate.

**Why this matters.** Roughly two-thirds of a transformer's parameters live in its
MLP blocks, so the MLP design is high-leverage. SwiGLU is the feed-forward of choice
in modern LLMs (LLaMA, PaLM) because it squeezes more quality out of the same compute
budget.

**The intuition.** The classic MLP is `Linear -> GELU -> Linear`: project up, apply a
fixed nonlinearity, project down. SwiGLU adds a *gate*, a second projection that acts
like a volume knob the network learns to turn. For each value flowing through, the
gate decides how much to let pass: turn it up to amplify the signal, turn it down to
suppress it. Because the gate multiplies the main signal (rather than just adding to
it), the network can express richer, content-dependent behavior.

**The mechanics.** SwiGLU computes `down( silu(gate(x)) * up(x) )`. There are three
linear projections now instead of two: `gate` and `up` both read the input, `silu`
(a smooth S-shaped activation) shapes the gate, the two are multiplied elementwise,
and `down` projects back to the model dimension. Three projections would normally cost
more *parameters* than two, so implementations shrink the hidden width (often to
about two-thirds) to hold the budget roughly equal. The benefit comes from the gating
mechanism, not from spending extra parameters.

In [ ]:
from pocketlm import SwiGLU
from torch import nn

dim = 16
swiglu = SwiGLU(dim)
gelu_mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))
x = torch.randn(2, 5, dim)
print("SwiGLU preserves shape:", tuple(swiglu(x).shape))
print("SwiGLU params:", sum(p.numel() for p in swiglu.parameters()),
      " GELU-MLP params:", sum(p.numel() for p in gelu_mlp.parameters()))

**What just happened.** The shape check confirms SwiGLU is a drop-in for the MLP: it
takes a `[batch, tokens, dim]` tensor and returns the same shape, so it slots into the
block wherever the old MLP sat. The parameter counts let you compare its budget against
the GELU MLP and see the three-projection-with-narrower-hidden tradeoff in numbers.

⚠️ **Common pitfalls**
- Counting the gate as free capacity. It is a third projection; if you do not shrink
  the hidden width you have quietly grown the model and any improvement may just be
  from the extra parameters, not the gating.
- Confusing the activations. The gate uses SiLU (also called swish), not GELU; the
  'Swi' in SwiGLU is exactly that choice, and the 'GLU' is the gated-linear-unit
  multiply.
- Forgetting that the multiply is elementwise. `gate(x)` and `up(x)` must have the
  same shape so they can be multiplied term by term.

🏋️ **Try it yourself.** Print the parameter counts of `SwiGLU(dim)` for a few values
of `dim` and compare against the GELU MLP. Notice how the gap scales, then think about
what hidden width would make the two exactly equal.

In [ ]:
assert tuple(swiglu(x).shape) == (2, 5, dim)